# Session 1 Suggested Exercises - Interview Training Notebook

Goal: build hands-on fluency for quant researcher interviews by practicing the exact topics from Session 1 in a structured order.

## How To Use This Notebook

1. Run cells in order.
2. Do not skip reflection prompts; they are interview rehearsal.
3. For each section, capture: speed, correctness, trade-offs, and production considerations.
4. Repeat weak sections until you can explain them without notes.

In [1]:
# Progress tracker
progress = {
    'env_git_docker': False,
    'timing_decorator': False,
    'pandas_benchmark': False,
    'binary_storage': False,
    'parallel_multiprocessing': False,
    'parallel_multithreading': False,
    'parallel_ray_local': False,
    'parallel_ray_remote': False,
    'scheduler_dag': False,
    'interview_drill': False,
}
progress

{'env_git_docker': False,
 'timing_decorator': False,
 'pandas_benchmark': False,
 'binary_storage': False,
 'parallel_multiprocessing': False,
 'parallel_multithreading': False,
 'parallel_ray_local': False,
 'parallel_ray_remote': False,
 'scheduler_dag': False,
 'interview_drill': False}

## Section 1: Environment, Git, Docker (Concept + Practice)

### Exercise
- Create a GitHub repo named `alpha-practice`.
- Write a `Dockerfile` that installs Miniconda and creates a Python environment.
- Build and run the image.

### Interview prompts
- Why does environment reproducibility matter in research?
- Difference between image and container?
- Why Git branching strategy matters for research velocity?

Mark `progress['env_git_docker'] = True` after completion.

In [ ]:
# Section 1 evidence: reproducibility, image vs container, git workflow
from pathlib import Path
import subprocess
import hashlib
import json

def run_cmd(cmd):
    try:
        out = subprocess.run(cmd, capture_output=True, text=True, check=True)
        return out.stdout.strip() or out.stderr.strip()
    except Exception as e:
        return f'Unavailable: {e}'

dockerfile_candidates = [Path('Dockerfile'), Path('./alpha-session1-practice/Dockerfile')]
dockerfile_path = next((p for p in dockerfile_candidates if p.exists()), None)

if dockerfile_path:
    docker_bytes = dockerfile_path.read_bytes()
    docker_sha = hashlib.sha256(docker_bytes).hexdigest()[:16]
else:
    docker_sha = None

evidence = {
    'git_version': run_cmd(['git', '--version']),
    'docker_version': run_cmd(['docker', '--version']),
    'dockerfile_found': str(dockerfile_path) if dockerfile_path else 'not found',
    'dockerfile_hash_prefix': docker_sha,
    'repo_state_example': run_cmd(['git', 'status', '--short']) if Path('.git').exists() else 'not a git repo in cwd',
}

print(json.dumps(evidence, indent=2))

section1_answer = {
    'Q1_reproducibility': 'Pinning dependencies and image hash makes research runs repeatable and auditable.',
    'Q2_image_vs_container': 'Image is immutable template; container is runtime instance with state.',
    'Q3_git_branching_velocity': 'Feature branches isolate experiments, reduce merge risk, and preserve research iteration speed.'
}
section1_answer

{
  "git_version": "git version 2.32.1 (Apple Git-133)",
  "docker_version": "Docker version 26.0.0, build 2ae903e",
  "dockerfile_found": "not found",
  "dockerfile_hash_prefix": null,
  "repo_state_example": "not a git repo in cwd"
}


{'Q1_reproducibility': 'Pinning dependencies and image hash makes research runs repeatable and auditable.',
 'Q2_image_vs_container': 'Image is immutable template; container is runtime instance with state.',
 'Q3_git_branching_velocity': 'Feature branches isolate experiments, reduce merge risk, and preserve research iteration speed.'}

In [5]:
import time
from functools import wraps

def timed(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        # TODO 1: 记录开始时间 (提示: 使用 time.perf_counter())
        # TODO 2: 运行目标函数 func(*args, **kwargs) 并保存结果
        # TODO 3: 记录结束时间，计算 elapsed 耗时，并打印出来
        # TODO 4: 返回函数结果
        pass
    return wrapper

@timed
def sample_work(n=1_000_000):
    s = 0
    for i in range(n):
        s += i
    return s

print("测试 Timing 装饰器 (如实现正确，应打印耗时):")
result = sample_work(200_000)
assert result == 19999900000

测试 Timing 装饰器 (如实现正确，应打印耗时):


AssertionError: 

In [8]:
import time
from functools import wraps

def timed(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f'{func.__name__} took {elapsed:.6f}s')
        return result
    return wrapper

@timed
def sample_work(n=1_000_000):
    s = 0
    for i in range(n):
        s += i
    return s

sample_work(200_000)

sample_work took 0.006877s


19999900000

### Interview prompts
- Why use `time.perf_counter()` instead of `time.time()`?
- What can make micro-benchmarks misleading?
- How would you benchmark fairly across implementations?

In [9]:
# Section 2 evidence: timer quality + benchmark reliability
import statistics
import time

def timer_resolution(timer_fn, n=50000):
    vals = []
    last = timer_fn()
    for _ in range(n):
        now = timer_fn()
        delta = now - last
        if delta > 0:
            vals.append(delta)
        last = now
    return {
        'samples': len(vals),
        'min_positive_delta_us': min(vals) * 1e6 if vals else None,
        'median_delta_us': statistics.median(vals) * 1e6 if vals else None,
    }

perf_stats = timer_resolution(time.perf_counter)
wall_stats = timer_resolution(time.time)
print('perf_counter stats:', perf_stats)
print('time.time stats:', wall_stats)

def micro_bench(fn, repeats=8):
    times = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        fn()
        times.append(time.perf_counter() - t0)
    return {
        'mean_ms': statistics.mean(times) * 1e3,
        'std_ms': statistics.stdev(times) * 1e3 if len(times) > 1 else 0.0,
        'raw_ms': [round(x * 1e3, 3) for x in times],
    }

bench_noise = micro_bench(lambda: sample_work(120_000), repeats=10)
print('benchmark variability (ms):', bench_noise)

section2_answer_card = {
    'Q1_perf_counter_vs_time': 'perf_counter usually provides higher-resolution monotonic timing for benchmarking.',
    'Q2_micro_benchmark_risk': 'Run-to-run variance exists; use repeats, warm-up, and robust summary stats.',
    'Q3_fair_benchmarking': 'Use identical input, same machine state, multiple repeats, and compare mean/std together.'
}
section2_answer_card

perf_counter stats: {'samples': 50000, 'min_positive_delta_us': 0.040978193283081055, 'median_delta_us': 0.12479722499847412}
time.time stats: {'samples': 4476, 'min_positive_delta_us': 0.7152557373046875, 'median_delta_us': 0.95367431640625}
sample_work took 0.004181s
sample_work took 0.004974s
sample_work took 0.004621s
sample_work took 0.004439s
sample_work took 0.004478s
sample_work took 0.004740s
sample_work took 0.004563s
sample_work took 0.004312s
sample_work took 0.004417s
sample_work took 0.004590s
benchmark variability (ms): {'mean_ms': 4.549454082734883, 'std_ms': 0.23019560134505804, 'raw_ms': [4.196, 5.014, 4.643, 4.447, 4.498, 4.757, 4.583, 4.32, 4.432, 4.605]}


{'Q1_perf_counter_vs_time': 'perf_counter usually provides higher-resolution monotonic timing for benchmarking.',
 'Q2_micro_benchmark_risk': 'Run-to-run variance exists; use repeats, warm-up, and robust summary stats.',
 'Q3_fair_benchmarking': 'Use identical input, same machine state, multiple repeats, and compare mean/std together.'}

In [ ]:
import time

def bench(name, fn):
    t0 = time.perf_counter()
    out = fn()
    dt = time.perf_counter() - t0
    # 安全检查：如果没有return则给出提示
    return {'method': name, 'seconds': dt, 'rows': len(out) if out is not None else 0}

def ffill_loop():
    out = df_sparse.copy()
    # TODO 1: 循环 df_sparse 的每一种 symbol
    # TODO 2: 提取当前 symbol 的布尔 mask，使用 .loc 取出只属于该 symbol 的 value列 进行 .ffill()
    return out

def ffill_apply():
    out = df_sparse.copy()
    # TODO 3: 使用 groupby 针对 'symbol' 维度，结合 apply 和 lambda 表达式做 ffill()
    return out

def ffill_groupby():
    out = df_sparse.copy()
    # TODO 4: 最优雅解法：直接对被 groupby 后的列对象调用 .ffill() 方法
    return out

def ffill_unstack_stack():
    # TODO 5: 将 'symbol' 维度 unstack 成宽表
    # TODO 6: 对宽表整体做 .ffill()
    # TODO 7: 再 stack 切回长表，注意参数 dropna=False 并用 to_frame('value') 将系列转换为与原表一致的结构，之后 sort_index()
    return None

results = [
    bench('loop', ffill_loop),
    bench('apply', ffill_apply),
    bench('groupby', ffill_groupby),
    bench('unstack_ffill_stack', ffill_unstack_stack),
]
pd.DataFrame(results).sort_values('seconds')

In [10]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
dates = pd.date_range('2024-01-01', periods=120, freq='B')
symbols = [f'STK{i:03d}' for i in range(60)]
idx = pd.MultiIndex.from_product([dates, symbols], names=['date', 'symbol'])
df = pd.DataFrame({'value': rng.normal(size=len(idx))}, index=idx).sort_index()

drop_n = int(0.1 * len(df))
to_drop = rng.choice(len(df), size=drop_n, replace=False)
df_sparse = df.drop(df.index[to_drop]).sort_index()

df.shape, df_sparse.shape

((7200, 1), (6480, 1))

In [13]:
import time

def bench(name, fn):
    t0 = time.perf_counter()
    out = fn()
    dt = time.perf_counter() - t0
    return {'method': name, 'seconds': dt, 'rows': len(out)}

def ffill_loop():
    out = df_sparse.copy()
    for sym in out.index.get_level_values('symbol').unique():
        m = out.index.get_level_values('symbol') == sym
        out.loc[m, 'value'] = out.loc[m, 'value'].ffill()
    return out

def ffill_apply():
    out = df_sparse.copy()
    out['value'] = out.groupby(level='symbol')['value'].apply(lambda x: x.ffill()).droplevel(0)
    return out

def ffill_groupby():
    out = df_sparse.copy()
    out['value'] = out.groupby(level='symbol')['value'].ffill()
    return out

def ffill_unstack_stack():
    wide = df_sparse['value'].unstack('symbol')
    wide = wide.ffill()
    #out = wide.stack(dropna=False).to_frame('value').sort_index()
    out = wide.stack().to_frame('value').sort_index()
    return out

results = [
    bench('loop', ffill_loop),
    bench('apply', ffill_apply),
    bench('groupby', ffill_groupby),
    bench('unstack_ffill_stack', ffill_unstack_stack),
]
pd.DataFrame(results).sort_values('seconds')

,method,seconds,rows
2,groupby,0.000973,6480
3,unstack_ffill_stack,0.001725,7200
1,apply,0.007970,6480
0,loop,0.076150,6480


In [ ]:
from pathlib import Path

base = Path('./session1_artifacts')
single_dir = base / 'single'
date_dir = base / 'shard_by_date'
symbol_dir = base / 'shard_by_symbol'
for d in [single_dir, date_dir, symbol_dir]:
    d.mkdir(parents=True, exist_ok=True)

single_path = single_dir / 'panel.parquet'
parquet_available = True

try:
    # TODO 1: 将 df_sparse 存为 parquet 格式到 single_path 路径
    pass
except Exception as e:
    parquet_available = False
    single_path = single_dir / 'panel.pkl'
    df_sparse.to_pickle(single_path)
    print('Parquet unavailable, fallback to pickle:', e)

print("检查单文件是否生成:", single_path.exists())
single_path, parquet_available

                method   seconds  rows
0              groupby  0.000973  6480
1  unstack_ffill_stack  0.001725  7200
2                apply  0.007970  6480
3                 loop  0.076150  6480
loop == groupby: True
groupby == unstack-stack: False
{'long_memory_bytes': 75920, 'wide_memory_bytes': 62728}


{'Q1_groupby_ffill_advantage': 'Vectorized groupby path usually reduces Python-loop overhead.',
 'Q2_when_unstack_faster': 'When dense-by-time matrix operations dominate and memory is sufficient.',
 'Q3_memory_tradeoff': 'Observed wide/long memory ratio = 0.83.'}

In [ ]:
# Shard by date and symbol
if parquet_available:
    # TODO 2: 按 'date' 维度 groupby 遍历 df_sparse
    # 提示: name格式 => date_dir / f'date={dt.date()}.parquet'
    # 对每个切片单独存为 parquet
    pass

    # TODO 3: 按 'symbol' 维度 groupby 遍历 df_sparse
    # 提示: name格式 => symbol_dir / f'symbol={sym}.parquet'
    # 对每个切片单独存为 parquet
    pass

    shard_counts = (len(list(date_dir.glob('*.parquet'))), len(list(symbol_dir.glob('*.parquet'))))
else:
    for dt, g in df_sparse.groupby(level='date'):
        out = date_dir / f'date={dt.date()}.pkl'
        g.to_pickle(out)

    for sym, g in df_sparse.groupby(level='symbol'):
        out = symbol_dir / f'symbol={sym}.pkl'
        g.to_pickle(out)

    shard_counts = (len(list(date_dir.glob('*.pkl'))), len(list(symbol_dir.glob('*.pkl'))))

print("检查分片数量 (预期 ~120 date / ~60 symbol):", shard_counts)
shard_counts

## Section 4: Binary Storage (single file + sharding)

In [15]:
from pathlib import Path

base = Path('./session1_artifacts')
single_dir = base / 'single'
date_dir = base / 'shard_by_date'
symbol_dir = base / 'shard_by_symbol'
for d in [single_dir, date_dir, symbol_dir]:
    d.mkdir(parents=True, exist_ok=True)

single_path = single_dir / 'panel.parquet'
parquet_available = True
try:
    df_sparse.to_parquet(single_path)
except Exception as e:
    parquet_available = False
    single_path = single_dir / 'panel.pkl'
    df_sparse.to_pickle(single_path)
    print('Parquet unavailable, fallback to pickle:', e)

single_path, parquet_available

Parquet unavailable, fallback to pickle: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - `Import pyarrow` failed. pyarrow is required for parquet support. Use pip or conda to install the pyarrow package.
 - `Import fastparquet` failed. fastparquet is required for parquet support. Use pip or conda to install the fastparquet package.


(PosixPath('session1_artifacts/single/panel.pkl'), False)

In [ ]:
# Prepare a day-wise input list
daily_inputs = [g for _, g in df_sparse.groupby(level='date')]

def daily_avg(day_df):
    return float(day_df['value'].mean())

# Sequential baseline
t0 = time.perf_counter()
# TODO 1: 使用列表推导式 (list comprehension) 将 daily_avg 应用到 daily_inputs 每一个元素上
baseline = [] 
baseline_seconds = time.perf_counter() - t0
print(f'sequential baseline took {baseline_seconds:.6f}s')

print("检查输出数量是否正确 (预期120):", len(baseline))
baseline[:3]

(120, 60)

In [ ]:
from multiprocessing import Pool, cpu_count
import time

workers = max(1, cpu_count() - 1)
try:
    t0 = time.perf_counter()
    # TODO 2: 使用 multiprocessing.Pool 上下文管理器, 并调用 p.map 映射 daily_avg 到 daily_inputs
    # 提示: with Pool(...) as p:
    #           mp_out = ...
    mp_out = []
    mp_seconds = time.perf_counter() - t0
    print(f'multiprocessing took {mp_seconds:.6f}s')
except Exception as e:
    mp_out = baseline[:]
    mp_seconds = float('nan')
    print('multiprocessing unavailable, fallback to baseline:', e)

print("MP 输出数量是否与 baseline 一致:", len(mp_out) == len(daily_inputs))
mp_out[:3]

{'stored_format': 'pickle', 'single_read_sec': 0.001412, 'date_shard_read_sec': 0.000267, 'symbol_shard_read_sec': 0.000178, 'stored_bytes': 67698, 'csv_bytes': 243871, 'csv_vs_stored_size_ratio': 3.6}


{'Q1_single_vs_shard': 'Shard when query pattern is selective and partition key is known.',
 'Q2_date_sharding_pattern': 'Date-focused backtests and daily pipelines benefit from date partitions.',
 'Q3_parquet_vs_csv': 'Columnar typed storage is usually smaller and faster than CSV for analytics; this run reports empirical size/read stats.'}

In [ ]:
from concurrent.futures import ThreadPoolExecutor
import time

t0 = time.perf_counter()
# TODO 3: 使用 ThreadPoolExecutor 上下文管理器, 并调用 ex.map 映射 daily_avg 到 daily_inputs
# 提示1: with ThreadPoolExecutor(...) as ex:
# 提示2: 注意 .map() 在 Executor 中返回的是生成器，需套一层 list() 转化
th_out = []
th_seconds = time.perf_counter() - t0
print(f'multithreading took {th_seconds:.6f}s')

print("TH 输出数量是否正确:", len(th_out) == len(daily_inputs))
th_out[:3]

## Section 5: Parallelization - Daily Average

Core task from suggested exercise: compute daily average and parallelize by day.

In [ ]:
# Ray local cluster (optional if ray is installed)
try:
    import ray

    if not ray.is_initialized():
        ray.init(ignore_reinit_error=True)

    # TODO 4: 使用 ray.remote 装饰器改装 daily_avg 函数
    # @ray.remote
    # def daily_avg_remote(day_df): ...
    
    t0 = time.perf_counter()
    # TODO 5: 利用 remote_func.remote() 提交任务形成 future 列表，再使用 ray.get() 收集结果
    ray_out = []
    ray_seconds = time.perf_counter() - t0
    print(f'ray local took {ray_seconds:.6f}s')
    print('ray local ok', len(ray_out), ray_out[:3])
except Exception as e:
    print('Ray local skipped:', e)

sequential baseline took 0.005347s


(120, [0.05625809163427797, -0.24286252611980905, -0.05300131850870139])

In [19]:
from multiprocessing import Pool, cpu_count
import time

workers = max(1, cpu_count() - 1)
try:
    t0 = time.perf_counter()
    with Pool(processes=workers) as p:
        mp_out = p.map(daily_avg, daily_inputs)
    mp_seconds = time.perf_counter() - t0
    print(f'multiprocessing took {mp_seconds:.6f}s')
except Exception as e:
    mp_out = baseline[:]
    mp_seconds = float('nan')
    print('multiprocessing unavailable, fallback to baseline:', e)

len(mp_out), mp_out[:3]

Process SpawnPoolWorker-1:
Traceback (most recent call last):
Process SpawnPoolWorker-2:
Process SpawnPoolWorker-3:
Traceback (most recent call last):
Traceback (most recent call last):
Process SpawnPoolWorker-4:
Traceback (most recent call last):
  File "/Users/huanghaotian/anaconda3/envs/course311/lib/python3.11/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/Users/huanghaotian/anaconda3/envs/course311/lib/python3.11/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/Users/huanghaotian/anaconda3/envs/course311/lib/python3.11/multiprocessing/pool.py", line 114, in worker
    task = get()
           ^^^^^
  File "/Users/huanghaotian/anaconda3/envs/course311/lib/python3.11/multiprocessing/queues.py", line 367, in get
    return _ForkingPickler.loads(res)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
AttributeError: Can't get attribute 'daily_avg' on <module '__main__' (built-in)>
  File "/Users/huanghaotian/anaconda3

KeyboardInterrupt: 

In [ ]:
from concurrent.futures import ThreadPoolExecutor
import time

t0 = time.perf_counter()
with ThreadPoolExecutor(max_workers=workers) as ex:
    th_out = list(ex.map(daily_avg, daily_inputs))
th_seconds = time.perf_counter() - t0
print(f'multithreading took {th_seconds:.6f}s')

len(th_out), th_out[:3]

### Interview prompts
- Why might multiprocessing beat multithreading for CPU-bound tasks in Python?
- What overhead can erase multiprocessing gains?
- What changes if the task becomes IO-bound?

In [ ]:
# Section 5 evidence: CPU-bound vs IO-bound behavior
import math

# Correctness check
print('baseline == multiprocessing:', np.allclose(baseline, mp_out))
print('baseline == multithreading:', np.allclose(baseline, th_out))

speed_table = pd.DataFrame([
    {'method': 'sequential', 'seconds': baseline_seconds},
    {'method': 'multiprocessing', 'seconds': mp_seconds},
    {'method': 'multithreading', 'seconds': th_seconds},
]).sort_values('seconds').reset_index(drop=True)
speed_table['speedup_vs_seq'] = baseline_seconds / speed_table['seconds']
print(speed_table)

# IO-bound toy task to show thread usefulness
def io_task(x):
    time.sleep(0.01)
    return x

io_inputs = list(range(120))
t0 = time.perf_counter()
_ = [io_task(x) for x in io_inputs]
io_seq = time.perf_counter() - t0

t0 = time.perf_counter()
with ThreadPoolExecutor(max_workers=workers) as ex:
    _ = list(ex.map(io_task, io_inputs))
io_th = time.perf_counter() - t0

print({'io_seq_sec': round(io_seq, 4), 'io_thread_sec': round(io_th, 4)})

section5_answer_card = {
    'Q1_cpu_bound_mp_vs_mt': 'CPU-bound tasks often favor multiprocessing due to GIL constraints.',
    'Q2_mp_overhead': 'Process spawn/serialization/scheduling overhead can erase gains for small tasks.',
    'Q3_io_bound_change': 'For IO-bound workloads, threads can outperform sequential due to wait overlap.'
}
section5_answer_card

In [ ]:
# Ray local cluster (optional if ray is installed)
try:
    import ray

    if not ray.is_initialized():
        ray.init(ignore_reinit_error=True)

    @ray.remote
    def daily_avg_remote(day_df):
        return float(day_df['value'].mean())

    t0 = time.perf_counter()
    ray_out = ray.get([daily_avg_remote.remote(x) for x in daily_inputs])
    ray_seconds = time.perf_counter() - t0
    print(f'ray local took {ray_seconds:.6f}s')
    print('ray local ok', len(ray_out), ray_out[:3])
except Exception as e:
    print('Ray local skipped:', e)

### Remote Ray cluster template

Use this when you have a remote head node: `ray.init(address='ray://<host>:10001')`

Interview prompt: local multiprocessing vs distributed ray, compare scheduling overhead, fault tolerance, and scaling limits.

In [ ]:
# Remote Ray interview evidence scaffold (runs safely without remote cluster)
remote_ray_observation = {}
try:
    import ray
    # This call is intentionally guarded; replace host when remote cluster is available.
    # ray.init(address='ray://<host>:10001')
    remote_ray_observation['remote_cluster_connected'] = False
    remote_ray_observation['note'] = 'Fill host and run in your infra to capture real numbers.'
except Exception as e:
    remote_ray_observation['remote_cluster_connected'] = False
    remote_ray_observation['note'] = f'Ray not available: {e}'

section5b_answer_card = {
    'Q4_local_vs_remote_ray': 'Remote Ray improves horizontal scalability and fault isolation, but adds network and scheduling overhead.'
}

print(remote_ray_observation)
section5b_answer_card

## Section 6: Scheduling Dependent Jobs

Goal: create a chain of simple jobs where each depends on previous completion.

In [ ]:
# Simple in-notebook dependency chain (conceptual DAG)
import time

state = {}

def job_extract():
    state['raw'] = [1, 2, 3, 4]
    print('extract done')

def job_transform():
    if 'raw' not in state:
        raise RuntimeError('extract must run first')
    state['features'] = [x * 10 for x in state['raw']]
    print('transform done')

def job_load():
    if 'features' not in state:
        raise RuntimeError('transform must run first')
    state['loaded'] = True
    print('load done')

job_extract()
job_transform()
job_load()
state

### Production extension
- Rebuild this flow in Airflow or Dagster as a real DAG.
- Add retries, alerting, and idempotency.

Interview prompts
- What does idempotent job design mean?
- How do you prevent partial writes from corrupting downstream jobs?

In [ ]:
# Section 6 evidence: idempotency + partial write protection
from pathlib import Path
import json
import tempfile
import os

pipeline_dir = Path('./session1_artifacts/pipeline')
pipeline_dir.mkdir(parents=True, exist_ok=True)
target = pipeline_dir / 'features.json'

def atomic_write_json(path, payload):
    fd, tmp_path = tempfile.mkstemp(dir=str(path.parent), prefix='tmp_', suffix='.json')
    os.close(fd)
    # TODO 1: 通过 with open(tmp_path) 的方式把 payload 转为 JSON 并写入 tmp_path
    # TODO 2: 使用 os.replace 原子化地将 tmp_path 重命名(覆盖)为目标 path
    pass

payload = {'version': 1, 'features': [10, 20, 30]}
atomic_write_json(target, payload)

# 防止未编写代码引发异常
if target.exists():
    before = target.read_text(encoding='utf-8')

    # Re-run with same payload (idempotent result expected)
    atomic_write_json(target, payload)
    after = target.read_text(encoding='utf-8')

    print('idempotent_content_equal (期待 True):', before == after)
    print('target_exists:', target.exists(), 'size:', target.stat().st_size)
else:
    print("请先完成上方 TODO 以创建目标文件")

section6_answer_card = {
    'Q1_idempotency': 'Repeated run with same input leaves final state unchanged.',
    'Q2_partial_write_safety': 'Write temp file then atomic replace to avoid downstream reading partial output.'
}
section6_answer_card

## Section 7: Interview Drill

Answer each in 60-90 seconds out loud:
1. Explain GIL impact on multiprocessing vs multithreading.
2. Explain your benchmark methodology and how you avoided bias.
3. Explain when to shard by date vs symbol.
4. Explain why reproducible environments are alpha-protective.
5. Explain a production-safe scheduled research pipeline.

In [ ]:
# Section 7 evidence: compile all answer cards into interview-ready script
answer_bank = {
    **section1_answer_card,
    **section2_answer_card,
    **section3_answer_card,
    **section4_answer_card,
    **section5_answer_card,
    **section5b_answer_card,
    **section6_answer_card,
}

for k in sorted(answer_bank):
    print(f'{k}: {answer_bank[k]}')

required_questions = [
    'Q1_reproducibility', 'Q2_image_vs_container', 'Q3_git_branching_velocity',
    'Q1_perf_counter_vs_time', 'Q2_micro_benchmark_risk', 'Q3_fair_benchmarking',
    'Q1_groupby_ffill_advantage', 'Q2_when_unstack_faster', 'Q3_memory_tradeoff',
    'Q1_single_vs_shard', 'Q2_date_sharding_pattern', 'Q3_parquet_vs_csv',
    'Q1_cpu_bound_mp_vs_mt', 'Q2_mp_overhead', 'Q3_io_bound_change',
    'Q4_local_vs_remote_ray',
    'Q1_idempotency', 'Q2_partial_write_safety',
]
missing = [q for q in required_questions if q not in answer_bank]
print('missing_answers:', missing if missing else 'None')

len(answer_bank), 'answers covered'

In [ ]:
# Final scorecard (self-evaluation)
scorecard = {
    'clarity': 0,          # 0-5
    'technical_depth': 0,  # 0-5
    'tradeoff_reasoning': 0,# 0-5
    'production_mindset': 0,# 0-5
    'speed_under_pressure': 0,# 0-5
}

total = sum(scorecard.values())
print('Total score / 25 =', total)
if total < 15:
    print('Action: repeat Sections 3-6 and re-drill interview answers.')
elif total < 21:
    print('Action: improve precision and shorten explanations.')
else:
    print('Action: mock interview ready. Keep daily repetition.')